In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
# set relative path and import data
data_path = Path("../../data/clean_data_tea")
geo_path = data_path / "geo_clean.csv"
wb_path = data_path / "wb_combined.csv"

geo = pd.read_csv(geo_path)
wb = pd.read_csv(wb_path)

In [3]:
print(geo.shape)

(252, 8)


In [4]:
# compare iso3 
geo_iso = set(geo["iso3"])
wb_iso = set(wb["iso3"])

geo_only = geo_iso - wb_iso
wb_only = wb_iso - geo_iso
common_iso = geo_iso & wb_iso

print(len(common_iso))


200


In [5]:
# check geo only iso
geo[geo["iso3"].isin(geo_only)][["iso3", "country"]]

,iso3,country
4,AIA,Anguilla
8,ATA,Antarctica
13,ABW,Aruba
14,ALA,Aland Islands
25,BLM,Saint Barthelemy
29,BES,"Bonaire, Saint Eustatius and Saba"
33,BVT,Bouvet Island
38,CCK,Cocos Islands
44,COK,Cook Islands
52,CUW,Curacao


In [6]:
# check world bank only iso
wb[wb["iso3"].isin(wb_only)][["iso3", "country"]].drop_duplicates()

,iso3,country
0,AFE,Eastern and Southern Africa
132,AFW,Western and Central Africa
396,ARB,Arab World
2310,CEB,Central Europe and the Baltics
3102,CSS,Caribbean small states
3762,EAP,East Asia & Pacific (excluding high income)
3828,EAR,Early-demographic dividend
3894,EAS,East Asia & Pacific
3960,ECA,Europe & Central Asia (excluding high income)
4026,ECS,Europe & Central Asia


I think it's better to keep only the common countries/territories between the datasets to keep it as clean as possible
- World Bank only: 48 -> aggregates that we don't need
- Geonames only: 52 -> territories/special entities/small counties/disputed territories 
- 200 common ones (for now) -> it's okay

In [7]:
# rename country column (wb) -> I don't know which one to keep
wb = wb.rename(columns={"country": "country_wb"})

In [8]:
# Final merge: inner bc we only keep the common iso3 (many to one bc geo has only one observation per iso)
merged = pd.merge(wb, geo, on="iso3", how="inner", validate="many_to_one")

In [9]:
print(merged.shape)
print(merged["iso3"].nunique())
print(merged.duplicated(subset=["iso3", "year"]).sum())
print(merged.isna().sum())

(13170, 13)
200
0
iso3                    0
country_wb              0
year                    0
population              0
forest_ratio         6323
pm2.5_pollution      6370
iso                    66
country                 0
area_sq_km              0
continent            1782
capital_latitude      102
capital_longitude     102
capital                 0
dtype: int64


In [10]:
# check iso missing values
# solve: just drop the column
merged[merged["iso"].isna()][["iso3", "country_wb", "country", "iso"]].drop_duplicates()

,iso3,country_wb,country,iso
8448,NAM,Namibia,Namibia,NaN


In [11]:
# check continent missing values
# solve: fix the import (because north america (NA), is read like a missing value) or fix it later
merged[merged["continent"].isna()][["iso3", "country_wb", "country"]].drop_duplicates()

,iso3,country_wb,country
528,ATG,Antigua and Barbuda,Antigua and Barbuda
1254,BHS,"Bahamas, The",Bahamas
1452,BLZ,Belize,Belize
1518,BMU,Bermuda,Bermuda
1716,BRB,Barbados,Barbados
2046,CAN,Canada,Canada
2772,CRI,Costa Rica,Costa Rica
2838,CUB,Cuba,Cuba
3168,DMA,Dominica,Dominica
3300,DOM,Dominican Republic,Dominican Republic


In [12]:
# check latitude missing values
# Solve: just keep them for now, maybe add coordinates manually later?
merged[merged["capital_latitude"].isna()][["iso3", "country", "capital"]].drop_duplicates()

,iso3,country,capital
5676,ISR,Israel,Jerusalem
9834,PSE,Palestinian Territory,East Jerusalem


In [13]:
# drop iso (iso2) column -> we already have iso3 
merged = merged.drop(columns="iso")

In [14]:
# fill in missing value for continent with "NA" = North America!!
merged["continent"] = merged["continent"].fillna("NA")

In [15]:
merged.isna().sum()

iso3                    0
country_wb              0
year                    0
population              0
forest_ratio         6323
pm2.5_pollution      6370
country                 0
area_sq_km              0
continent               0
capital_latitude      102
capital_longitude     102
capital                 0
dtype: int64

In [16]:
# check differences in country names to see what to keep in the final dataset
merged[merged["country_wb"] != merged["country"]][["iso3", "country_wb", "country"]].drop_duplicates()

,iso3,country_wb,country
1254,BHS,"Bahamas, The",Bahamas
1782,BRN,Brunei Darussalam,Brunei
2310,CIV,Côte d’Ivoire,Ivory Coast
2442,COD,"Congo, Dem. Rep.",Democratic Republic of the Congo
2508,COG,"Congo, Rep.",Republic of the Congo
3498,EGY,"Egypt, Arab Rep.",Egypt
4026,FSM,"Micronesia, Fed. Sts.",Micronesia
4422,GMB,"Gambia, The",Gambia
5478,IRN,"Iran, Islamic Rep.",Iran
6138,KGZ,Kyrgyz Republic,Kyrgyzstan


In [17]:
# keep only country and drop country_wb -> much cleaner names
merged = merged.drop(columns="country_wb")

In [20]:
# add population density column
merged["pop_density"] = (merged["population"] / merged["area_sq_km"]).round(2)

In [22]:
# export 
output_path = data_path / "geo_merged.csv"
merged.to_csv(output_path, index=False)